In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import pandas as pd, numpy as np
from pathlib import Path

DATA = next(Path("/kaggle/input").rglob("train_sequences.csv")).parent
print(DATA)
print(sorted(p.name for p in DATA.iterdir()))

/kaggle/input/competitions/stanford-rna-3d-folding
['MSA', 'MSA_v2', 'PDB_RNA', 'sample_submission.csv', 'test_sequences.csv', 'train_labels.csv', 'train_labels.v2.csv', 'train_sequences.csv', 'train_sequences.v2.csv', 'validation_labels.csv', 'validation_sequences.csv']


In [3]:
s1 = pd.read_csv(DATA/"train_sequences.csv")
s2 = pd.read_csv(DATA/"train_sequences.v2.csv")
for name, d in [("v1", s1), ("v2", s2)]:
    print(name, d.shape, d.columns.tolist())
    print("  cutoff range:", d["temporal_cutoff"].min(), "to", d["temporal_cutoff"].max())

ids1, ids2 = set(s1.target_id), set(s2.target_id)
print("only in v1:", len(ids1 - ids2), "| only in v2:", len(ids2 - ids1), "| shared:", len(ids1 & ids2))

# for shared targets, did the sequence or date change?
m = s1.merge(s2, on="target_id", suffixes=("_v1", "_v2"))
print("sequence changed:", (m.sequence_v1 != m.sequence_v2).sum())
print("cutoff changed:", (m.temporal_cutoff_v1 != m.temporal_cutoff_v2).sum())

l1 = pd.read_csv(DATA/"train_labels.csv", nrows=5)
l2 = pd.read_csv(DATA/"train_labels.v2.csv", nrows=5)
print(l1.columns.tolist()); print(l2.columns.tolist())

v1 (844, 5) ['target_id', 'sequence', 'temporal_cutoff', 'description', 'all_sequences']
  cutoff range: 1995-01-26 to 2024-12-18
v2 (5135, 5) ['target_id', 'sequence', 'temporal_cutoff', 'description', 'all_sequences']
  cutoff range: 1987-11-06 to 2025-03-26
only in v1: 244 | only in v2: 4535 | shared: 600
sequence changed: 0
cutoff changed: 0
['ID', 'resname', 'resid', 'x_1', 'y_1', 'z_1']
['ID', 'resname', 'resid', 'x_1', 'y_1', 'z_1']


In [4]:
# 1) what are the 244 targets that are only in v1?
only1 = s1[~s1.target_id.isin(s2.target_id)]
print(only1[["target_id","temporal_cutoff","description"]].head(10).to_string())
# are their sequences present in v2 under a different ID?
print("v1-only sequences found in v2:", only1.sequence.isin(s2.sequence).sum(), "of", len(only1))

# 2) how redundant is v2?
print("v2 targets:", len(s2), "| unique sequences:", s2.sequence.nunique())

# 3) what do the new v2 targets look like?
new = s2[~s2.target_id.isin(s1.target_id)]
print(new[["target_id","temporal_cutoff","description"]].head(10).to_string())
print(new.sequence.str.len().describe())

# 4) do the v2 labels cover the v2 targets, and what are unresolved coordinates?
l2 = pd.read_csv(DATA/"train_labels.v2.csv")
l2["target_id"] = l2["ID"].str.rsplit("_", n=1).str[0]
print("label targets:", l2.target_id.nunique(), "| seq targets:", len(s2))
print("targets missing labels:", len(set(s2.target_id) - set(l2.target_id)))
print(l2[["x_1","y_1","z_1"]].describe().T[["min","max"]])

   target_id temporal_cutoff                                                                                                                                       description
8     1SLO_A      1996-12-07                                                      FIRST STEM LOOP OF THE SL1 RNA FROM CAENORHABDITIS ELEGANS, NMR, MINIMIZED AVERAGE STRUCTURE
9     1BIV_A      1996-12-23                                                                                  BOVINE IMMUNODEFICIENCY VIRUS TAT-TAR COMPLEX, NMR, 5 STRUCTURES
10    1ANR_A      1997-01-11                                                                                 CIS-ACTING RNA REGULATORY ELEMENT (HIV-1 TAR), NMR, 20 STRUCTURES
19    1EBQ_A      1997-07-07                                                                         STRUCTURE OF RNA (5'-GGUGGGCGCAGCUUCGGCUGCGGUACCAC-3'), NMR, 5 STRUCTURES
26    1VOP_A      1997-12-24                                                                    CONSERVED RNA COMPONENT OF TH

In [5]:
s = s2.copy()
s["length"] = s.sequence.str.len()
s["pdb_id"] = s.target_id.str.split("_").str[0]

# 1) how many survive each length window?
for lo, hi in [(20,512),(10,200),(20,1000),(10,5000)]:
    n = s.length.between(lo, hi).sum()
    print(f"length {lo}-{hi}: {n} targets, {s[s.length.between(lo,hi)].sequence.nunique()} unique sequences")

# 2) non-ACGU letters
print("non-ACGU:", (~s.sequence.str.fullmatch(r"[ACGU]+")).sum())

# 3) how many ribosome-related descriptions (keyword is only a rough flag)
rib = s.description.str.contains("ribosom|rRNA|tRNA", case=False, na=False)
print("ribosome/rRNA/tRNA in description:", rib.sum(), "| of these within 20-512:", (rib & s.length.between(20,512)).sum())

# 4) chains per PDB entry
cnt = s.groupby("pdb_id").size().sort_values(ascending=False)
print(cnt.head(10)); print("entries:", len(cnt))

# 5) do any target IDs have more than one underscore?
print("IDs with >1 underscore:", (s.target_id.str.count("_") > 1).sum())

# 6) check 4 from before
l2 = pd.read_csv(DATA/"train_labels.v2.csv")
l2["target_id"] = l2["ID"].str.rsplit("_", n=1).str[0]
print("label targets:", l2.target_id.nunique(), "| seq targets:", len(s2))
print("targets missing labels:", len(set(s2.target_id) - set(l2.target_id)))
print(l2[["x_1","y_1","z_1"]].describe().T[["min","max"]])

# 7) PDB_RNA
files = sorted((DATA/"PDB_RNA").iterdir())
print(len(files), [f.name for f in files[:10]])

length 20-512: 3166 targets, 2331 unique sequences
length 10-200: 3316 targets, 2483 unique sequences
length 20-1000: 3280 targets, 2424 unique sequences
length 10-5000: 5135 targets, 4214 unique sequences
non-ACGU: 104
ribosome/rRNA/tRNA in description: 2708 | of these within 20-512: 1323
pdb_id
9G6K    25
6ZJ3    18
9FIA    11
8A3W    10
8A98    10
8RXX    10
5IBB     9
7PKT     9
2OM7     9
4V8M     9
dtype: int64
entries: 3800
IDs with >1 underscore: 0
label targets: 5135 | seq targets: 5135
targets missing labels: 0
            min          max
x_1 -942.017029   782.434021
y_1 -799.427979   889.507996
z_1 -465.916992  1160.218994
8672 ['124d.cif', '157d.cif', '165d.cif', '176d.cif', '17ra.cif', '1a1t.cif', '1a34.cif', '1a3m.cif', '1a4d.cif', '1a4t.cif']


In [6]:
# 6) labels: read only what's needed, to avoid memory problems
l2 = pd.read_csv(DATA/"train_labels.v2.csv", usecols=["ID","x_1","y_1","z_1"])
l2["target_id"] = l2["ID"].str.rsplit("_", n=1).str[0]
print("label rows:", len(l2), "| label targets:", l2.target_id.nunique(), "| seq targets:", len(s2))
print("targets missing labels:", len(set(s2.target_id) - set(l2.target_id)))
print("targets with labels but no sequence:", len(set(l2.target_id) - set(s2.target_id)))
print(l2[["x_1","y_1","z_1"]].describe().T[["min","max"]])
print("NaN coordinates:", l2[["x_1","y_1","z_1"]].isna().sum().to_dict())

label rows: 3677095 | label targets: 5135 | seq targets: 5135
targets missing labels: 0
targets with labels but no sequence: 0
            min          max
x_1 -942.017029   782.434021
y_1 -799.427979   889.507996
z_1 -465.916992  1160.218994
NaN coordinates: {'x_1': 39025, 'y_1': 39025, 'z_1': 39025}


In [7]:
# 7) PDB_RNA, in its own cell
p = DATA/"PDB_RNA"
files = sorted(p.iterdir())
print(len(files), [f.name for f in files[:10]])
print(sorted({f.suffix for f in files}))

8672 ['124d.cif', '157d.cif', '165d.cif', '176d.cif', '17ra.cif', '1a1t.cif', '1a34.cif', '1a3m.cif', '1a4d.cif', '1a4t.cif']
['.cif', '.csv', '.fasta']


In [8]:
s = pd.read_csv(DATA/"train_sequences.v2.csv")
l = pd.read_csv(DATA/"train_labels.v2.csv", usecols=["ID","x_1"])
l["target_id"] = l["ID"].str.rsplit("_", n=1).str[0]
l["resolved"] = l["x_1"].notna()

# sanity: label rows should equal the sum of sequence lengths
print("label rows:", len(l), "| sum of lengths:", s.sequence.str.len().sum())

frac = l.groupby("target_id")["resolved"].mean().rename("frac_resolved")
s = s.merge(frac, left_on="target_id", right_index=True, how="left")
s["length"] = s.sequence.str.len()
s["pdb_id"] = s.target_id.str.split("_").str[0].str.lower()

MIN_LEN, MAX_LEN, MIN_RES = 20, 512, 0.8
keep = (s.sequence.str.fullmatch(r"[ACGU]+")
        & s.length.between(MIN_LEN, MAX_LEN)
        & (s.frac_resolved >= MIN_RES))
print("after filters:", keep.sum())
c = (s[keep].sort_values("temporal_cutoff")
       .drop_duplicates("sequence", keep="first")
       .reset_index(drop=True))
print("after exact dedup:", len(c))

label rows: 3677095 | sum of lengths: 3677095
after filters: 2724
after exact dedup: 2107


In [9]:
from collections import defaultdict, Counter
import numpy as np

K, MIN_CONTAIN = 7, 0.2     # my choices; low threshold merges more, which is safer against leakage
seqs = c.sequence.tolist()
kmers = [set(q[i:i+K] for i in range(len(q)-K+1)) for q in seqs]
index = defaultdict(list)
for i, ks in enumerate(kmers):
    for k in ks: index[k].append(i)

parent = list(range(len(seqs)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for i, ks in enumerate(kmers):
    cnt = Counter()
    for k in ks:
        for j in index[k]:
            if j > i: cnt[j] += 1
    for j, n in cnt.items():
        if n / min(len(ks), len(kmers[j])) >= MIN_CONTAIN:
            parent[find(i)] = find(j)

c["family"] = [find(i) for i in range(len(c))]
sizes = c.family.value_counts()
print("clusters:", len(sizes)); print(sizes.head(10))

clusters: 200
family
2080    1823
608        5
1363       5
1978       5
1601       5
1077       4
199        4
450        4
2051       4
1019       3
Name: count, dtype: int64


In [10]:
rng = np.random.default_rng(0)
N = len(c); big = sizes[sizes > 0.05*N].index          # giant clusters forced into train
others = [f for f in sizes.index if f not in set(big)]
rng.shuffle(others)

assign, nv, nt = {f: "train" for f in big}, 0, 0
for f in others:
    if nv < 0.1*N:   assign[f] = "val";  nv += sizes[f]
    elif nt < 0.1*N: assign[f] = "test"; nt += sizes[f]
    else:            assign[f] = "train"
c["split"] = c.family.map(assign)
print(c.split.value_counts())

c = c.rename(columns={"target_id":"chain_id","temporal_cutoff":"release_date"})
c[["chain_id","family","split","release_date"]].to_csv("/kaggle/working/splits.tsv", sep="\t", index=False)
c[["chain_id","pdb_id","sequence","length","release_date","frac_resolved"]].to_csv("/kaggle/working/chain_list.tsv", sep="\t", index=False)

# leakage check: no family in two splits
print("families in >1 split:", (c.groupby("family").split.nunique() > 1).sum())

split
train    1823
val       211
test       73
Name: count, dtype: int64
families in >1 split: 0


In [11]:
p = DATA/"PDB_RNA"
files = list(p.iterdir())
print(Counter(f.suffix for f in files))

cif = next(f for f in files if f.suffix == ".cif")
txt = cif.read_text()
print(cif.name, len(txt), "chars")
print("has N9/N1 base atoms:", " N9 " in txt or " N1 " in txt, "| has C1':", "C1'" in txt)

csv_f = next(f for f in files if f.suffix == ".csv")
print(csv_f.name); print(pd.read_csv(csv_f, nrows=3))

have = {f.stem.lower() for f in files if f.suffix == ".cif"}
print("my chains whose entry has a .cif:", c.pdb_id.isin(have).mean())

Counter({'.cif': 8670, '.csv': 1, '.fasta': 1})
2d19.cif 1107746 chars
has N9/N1 base atoms: True | has C1': True
pdb_release_dates_NA.csv
  Entry ID Release Date
0     124D   1993-10-31
1     157D   1994-05-31
2     165D   1994-08-31
my chains whose entry has a .cif: 0.9558614143331752


In [12]:
# for each test chain: highest 7-mer containment against any train chain
tr = c.index[c.split == "train"].tolist()
te = c.index[c.split == "test"].tolist()
def best_overlap(i):
    cnt = Counter()
    for k in kmers[i]:
        for j in index[k]:
            if c.split[j] == "train": cnt[j] += 1
    return max((n / min(len(kmers[i]), len(kmers[j])) for j, n in cnt.items()), default=0)
ov = pd.Series([best_overlap(i) for i in te])
print(ov.describe()); print("test chains with overlap > 0.5 to train:", (ov > 0.5).sum(), "of", len(te))

count    73.000000
mean      0.159139
std       0.024265
min       0.090909
25%       0.142857
50%       0.161290
75%       0.180952
max       0.190476
dtype: float64
test chains with overlap > 0.5 to train: 0 of 73


In [13]:
import pandas as pd, numpy as np
from pathlib import Path
from collections import defaultdict, Counter

DATA = next(Path("/kaggle/input").rglob("train_sequences.v2.csv")).parent

# ---- load and filter ----
s = pd.read_csv(DATA/"train_sequences.v2.csv")
l = pd.read_csv(DATA/"train_labels.v2.csv", usecols=["ID","x_1"])
l["target_id"] = l["ID"].str.rsplit("_", n=1).str[0]
frac = l.assign(resolved=l.x_1.notna()).groupby("target_id")["resolved"].mean().rename("frac_resolved")
s = s.merge(frac, left_on="target_id", right_index=True, how="left")
s["length"] = s.sequence.str.len()
s["pdb_id"] = s.target_id.str.split("_").str[0].str.lower()

MIN_LEN, MAX_LEN, MIN_RES = 20, 512, 0.8
keep = (s.sequence.str.fullmatch(r"[ACGU]+")
        & s.length.between(MIN_LEN, MAX_LEN) & (s.frac_resolved >= MIN_RES))
c = (s[keep].sort_values("temporal_cutoff")
       .drop_duplicates("sequence", keep="first").reset_index(drop=True))
print("chains after filters and exact dedup:", len(c))

# ---- k-mer sets ----
K = 7
seqs = c.sequence.tolist()
kmers = [set(q[i:i+K] for i in range(len(q)-K+1)) for q in seqs]

# ---- greedy centroid clustering ----
THRESH = 0.5     # my choice; a similarity proxy, not sequence identity
order = sorted(range(len(seqs)), key=lambda i: -len(seqs[i]))
centroids, label = [], {}
cent_index = defaultdict(list)
for i in order:
    cnt = Counter()
    for k in kmers[i]:
        for cid in cent_index[k]: cnt[cid] += 1
    best = None
    for cid, n in cnt.items():
        if n / min(len(kmers[i]), len(kmers[centroids[cid]])) >= THRESH:
            if best is None or n > cnt[best]: best = cid
    if best is None:
        best = len(centroids); centroids.append(i)
        for k in kmers[i]: cent_index[k].append(best)
    label[i] = best
c["family"] = [label[i] for i in range(len(c))]
sizes = c.family.value_counts()
print("clusters:", len(sizes), "| 8 largest:", sizes.head(8).tolist(),
      "| singletons:", (sizes == 1).sum())

chains after filters and exact dedup: 2107
clusters: 1049 | 8 largest: [44, 29, 25, 23, 22, 17, 16, 16] | singletons: 664


In [14]:
# ---- split whole clusters ----
rng = np.random.default_rng(0)
N = len(c)
big = set(sizes[sizes > 0.05*N].index)
others = [f for f in sizes.index if f not in big]
rng.shuffle(others)
assign, nv, nt = {f: "train" for f in big}, 0, 0
for f in others:
    if nv < 0.1*N:   assign[f] = "val";  nv += sizes[f]
    elif nt < 0.1*N: assign[f] = "test"; nt += sizes[f]
    else:            assign[f] = "train"
c["split"] = c.family.map(assign)
print(c.split.value_counts())
print("families in >1 split:", (c.groupby("family").split.nunique() > 1).sum())

# ---- overlap check (same proxy as the clustering, so it only confirms the rule was applied) ----
index = defaultdict(list)
for i, ks in enumerate(kmers):
    for k in ks: index[k].append(i)
is_train = (c.split == "train").to_numpy()
def best_overlap(i):
    cnt = Counter()
    for k in kmers[i]:
        for j in index[k]:
            if is_train[j]: cnt[j] += 1
    return max((n / min(len(kmers[i]), len(kmers[j])) for j, n in cnt.items()), default=0.0)
for sp in ["val", "test"]:
    ov = pd.Series([best_overlap(i) for i in c.index[c.split == sp]])
    print(sp, ov.describe()[["mean","50%","max"]].round(2).to_dict(),
          "| >0.5:", (ov > 0.5).sum(), "of", len(ov))

# ---- release-date check ----
rd = pd.read_csv(DATA/"PDB_RNA"/"pdb_release_dates_NA.csv")
rd["pdb_id"] = rd["Entry ID"].str.lower()
rd["rel"] = pd.to_datetime(rd["Release Date"])
chk = c.merge(rd[["pdb_id","rel"]], on="pdb_id", how="left")
d = (pd.to_datetime(chk.temporal_cutoff) - chk.rel).dt.days
print("no date in csv:", chk.rel.isna().sum(), "| exact match:", round((d == 0).mean(), 3))

# ---- tool check ----
import shutil
for t in ["cd-hit-est", "mmseqs", "USalign", "US-align"]:
    print(t, "->", shutil.which(t))

split
train    1680
test      214
val       213
Name: count, dtype: int64
families in >1 split: 0
val {'mean': 0.4, '50%': 0.35, 'max': 1.0} | >0.5: 41 of 213
test {'mean': 0.29, '50%': 0.25, 'max': 1.0} | >0.5: 13 of 214


ParserError: Error tokenizing data. C error: Expected 2 fields in line 2502, saw 3


In [15]:
ov_all = pd.Series({i: best_overlap(i) for i in c.index[c.split != "train"]})
flag = ov_all[ov_all > 0.5].index
print("flagged:", c.loc[flag, "split"].value_counts().to_dict())
print("flagged length:\n", c.loc[flag, "length"].describe())
print("all-chains length:\n", c.length.describe())

c.loc[flag, "split"] = "excluded"
print(c.split.value_counts())

flagged: {'val': 41, 'test': 13}
flagged length:
 count     54.000000
mean      87.740741
std      119.550603
min       20.000000
25%       32.250000
50%       35.000000
75%       76.000000
max      478.000000
Name: length, dtype: float64
all-chains length:
 count    2107.000000
mean       88.850024
std        80.771060
min        20.000000
25%        36.000000
50%        71.000000
75%       103.000000
max       498.000000
Name: length, dtype: float64
split
train       1680
test         201
val          172
excluded      54
Name: count, dtype: int64


In [16]:
p = DATA/"PDB_RNA"/"pdb_release_dates_NA.csv"
lines = p.read_text().splitlines()
print(len(lines), lines[:3])
bad = [(i, ln) for i, ln in enumerate(lines) if ln.count(",") != 1]
print(len(bad), bad[:10])

rd = pd.read_csv(p, on_bad_lines="skip", engine="python")
print("rows read:", len(rd), "of", len(lines) - 1)

rd["pdb_id"] = rd["Entry ID"].str.lower()
rd["rel"] = pd.to_datetime(rd["Release Date"], errors="coerce")
chk = c.merge(rd[["pdb_id","rel"]], on="pdb_id", how="left")
d = (pd.to_datetime(chk.temporal_cutoff) - chk.rel).dt.days
print("no date in csv:", chk.rel.isna().sum(), "| exact match:", round((d == 0).mean(), 3))
print(d.describe())

import shutil
for t in ["cd-hit-est", "mmseqs", "USalign", "US-align"]:
    print(t, "->", shutil.which(t))

8686 ['Entry ID,Release Date', '"124D","1993-10-31"', '"157D","1994-05-31"']
6 [(2501, 'Identifier,StructureData,'), (2502, 'Entry ID,Release Date,'), (5003, 'Identifier,StructureData,'), (5004, 'Entry ID,Release Date,'), (7505, 'Identifier,StructureData,'), (7506, 'Entry ID,Release Date,')]
rows read: 8679 of 8685
no date in csv: 89 | exact match: 0.958
count    2018.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
dtype: float64
cd-hit-est -> None
mmseqs -> None
USalign -> None
US-align -> None


In [17]:
# 1) are the 89 undated chains the same as the chains with no .cif?
have_cif = {f.stem.lower() for f in (DATA/"PDB_RNA").iterdir() if f.suffix == ".cif"}
dated = set(rd.pdb_id)
c["has_cif"] = c.pdb_id.isin(have_cif)
c["has_date"] = c.pdb_id.isin(dated)
print(pd.crosstab(c.has_cif, c.has_date))

# 2) length by split
print(c.groupby("split").length.describe()[["count","25%","50%","75%","max"]])

# 3) leakage re-check after exclusions
is_train = (c.split == "train").to_numpy()
for sp in ["val", "test"]:
    ov = pd.Series([best_overlap(i) for i in c.index[c.split == sp]])
    print(sp, "max overlap:", round(ov.max(), 2), "| >0.5:", (ov > 0.5).sum())

# 4) how many val/test chains lack a structure file
print(c[c.split.isin(["val","test"])].groupby("split").has_cif.value_counts())

has_date  False  True 
has_cif               
False        89      4
True          0   2014
           count    25%   50%     75%    max
split                                       
excluded    54.0  32.25  35.0   76.00  478.0
test       201.0  33.00  61.0  112.00  470.0
train     1680.0  37.00  72.0  102.00  498.0
val        172.0  43.75  75.0  108.25  413.0
val max overlap: 0.5 | >0.5: 0
test max overlap: 0.5 | >0.5: 0
split  has_cif
test   True       188
       False       13
val    True       166
       False        6
Name: count, dtype: int64


In [18]:
for sp in ["val", "test"]:
    ov = pd.Series([best_overlap(i) for i in c.index[c.split == sp]])
    print(sp, "overlap >= 0.4:", (ov >= 0.4).sum(), "| >= 0.5:", (ov >= 0.5).sum(), "of", len(ov))

val overlap >= 0.4: 45 | >= 0.5: 6 of 172
test overlap >= 0.4: 16 | >= 0.5: 5 of 201


In [19]:
# 1) exclude >= 0.5
ov_by_idx = {i: best_overlap(i) for i in c.index[c.split.isin(["val","test"])]}
ov_s = pd.Series(ov_by_idx)
c.loc[ov_s[ov_s >= 0.5].index, "split"] = "excluded"

# 2) mark the 0.4-0.5 chains instead of removing them
c["borderline"] = False
c.loc[ov_s[(ov_s >= 0.4) & (ov_s < 0.5)].index, "borderline"] = True
print(c.split.value_counts())
print(c[c.split.isin(["val","test"])].groupby("split").borderline.sum())

split
train       1680
test         196
val          166
excluded      65
Name: count, dtype: int64
split
test    11
val     39
Name: borderline, dtype: int64


In [20]:
from pathlib import Path

# ---- 1) overlap of every val/test chain to the train set (train is unchanged so far) ----
is_train = (c.split == "train").to_numpy()
vt = c.index[c.split.isin(["val", "test"])]
ov = pd.Series({i: best_overlap(i) for i in vt})

# ---- 2) reasons for exclusion ----
c["reason"] = ""
c.loc[c.split == "excluded", "reason"] = "overlap_with_train"      # the 54 excluded earlier
c.loc[ov[ov >= 0.5].index, "reason"] = "overlap_with_train"         # newly excluded (>= 0.5)

have_cif = {f.stem.lower() for f in (DATA/"PDB_RNA").iterdir() if f.suffix == ".cif"}
c["has_cif"] = c.pdb_id.isin(have_cif)
c["has_date"] = c.pdb_id.isin(set(rd.pdb_id))
c.loc[(c.reason == "") & ~c.has_cif, "reason"] = "no_cif"
c.loc[(c.reason == "") & ~c.has_date, "reason"] = "no_date"
c.loc[c.reason != "", "split"] = "excluded"

# ---- 3) mark borderline chains (0.4 <= overlap < 0.5), kept in the split ----
c["borderline"] = False
c.loc[ov[(ov >= 0.4) & (ov < 0.5)].index, "borderline"] = True
c.loc[c.split == "excluded", "borderline"] = False

# ---- 4) write files ----
OUT = Path("/kaggle/working")
c2 = c.rename(columns={"target_id": "chain_id", "temporal_cutoff": "release_date"})
c2[["chain_id", "family", "split", "release_date", "reason", "borderline"]] \
    .to_csv(OUT/"splits.tsv", sep="\t", index=False)
c2[["chain_id", "pdb_id", "sequence", "length", "release_date", "frac_resolved", "has_cif"]] \
    .to_csv(OUT/"chain_list.tsv", sep="\t", index=False)

# ---- 5) checks on the files as saved ----
chk = pd.read_csv(OUT/"splits.tsv", sep="\t")
print(chk.split.value_counts())
print(chk.reason.fillna("").value_counts())
print("borderline per split:", chk[chk.split.isin(["val", "test"])].groupby("split").borderline.sum().to_dict())
print("clusters in >1 split (excluded ignored):",
      (chk[chk.split != "excluded"].groupby("family").split.nunique() > 1).sum())
print("duplicate chain_ids:", chk.chain_id.duplicated().sum())
print(sorted(p.name for p in OUT.iterdir()))

split
train       1606
test         184
val          160
excluded     157
Name: count, dtype: int64
reason
                      1950
no_cif                  92
overlap_with_train      65
Name: count, dtype: int64
borderline per split: {'test': 11, 'val': 37}
clusters in >1 split (excluded ignored): 0
duplicate chain_ids: 0
['.virtual_documents', 'chain_list.tsv', 'splits.tsv']
